In [2]:
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from IPython.display import Video, display
import robosuite.utils.transform_utils as T
from scipy.spatial.transform import Rotation as R
import imageio, os, numpy as np

suite_name = "libero_goal"
task_idx = 3
task_suite = benchmark.get_benchmark_dict()[suite_name]()
task = task_suite.get_task(task_idx)
bddl_file = os.path.join(
    get_libero_path("bddl_files"), task.problem_folder, task.bddl_file
)
print(task.name)

/home/richard/miniconda3/envs/libero/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /home/richard/miniconda3/envs/libero/lib/python3.8/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[Warning]: datasets path /home/richard/workspaces/cs7150/cs7150_diffusion_policy/submodules/LIBERO/libero/libero/../datasets does not exist!
open_the_top_drawer_and_put_the_bowl_inside


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
def make_env(delta=True):
    env = OffScreenRenderEnv(
        bddl_file_name=bddl_file, camera_heights=256, camera_widths=256
    )
    env.seed(0)
    obs = env.reset()
    for _ in range(10):
        obs, _, _, _ = env.step(np.zeros(7))
    if not delta:
        for robot in env.env.robots:
            robot.controller.use_delta = False
    print(f"use_delta={env.env.robots[0].controller.use_delta}")
    return env, obs


def eef(obs):
    return np.asarray(obs["robot0_eef_pos"]).flatten()


def obs_to_action(env, obs, target_pos=None):
    """Build 7D absolute action.
    robot0_eef_quat uses a different body/frame than the OSC controller's ee_ori_mat,
    so convert directly from the controller's rotation matrix to avoid any mismatch."""
    pos = eef(obs) if target_pos is None else target_pos
    ori_mat = env.env.robots[0].controller.ee_ori_mat
    aa = T.quat2axisangle(T.mat2quat(ori_mat))
    return np.concatenate([pos, aa, [1.0]])


def get_frame(obs):
    return obs["agentview_image"][::-1]


def show_video(frames, path="/tmp/libero_test.mp4", fps=20):
    imageio.mimsave(path, frames, fps=fps)
    display(Video(path, embed=True, width=400))

In [4]:
# Inspect controller + confirm quaternion convention
env, obs = make_env(delta=False)
ctrl = env.env.robots[0].controller
print(f"Controller : {type(ctrl).__name__}")
print(f"use_delta  : {ctrl.use_delta}")
print(f"input_max  : {ctrl.input_max}")
print(f"input_min  : {ctrl.input_min}")
print(f"output_max : {ctrl.output_max}")
print(f"output_min : {ctrl.output_min}")

q_obs = np.asarray(obs["robot0_eef_quat"]).flatten()
aa_direct = T.quat2axisangle(q_obs)  # treat as [x,y,z,w]
aa_reorder = T.quat2axisangle(
    np.array([q_obs[1], q_obs[2], q_obs[3], q_obs[0]])
)  # treat as [w,x,y,z]
aa_from_mat = T.quat2axisangle(
    T.mat2quat(ctrl.ee_ori_mat)
)  # ground truth from rotation matrix


print(f"\nrobot0_eef_quat  : {q_obs}")
print(
    f"aa direct [x,y,z,w]: {aa_direct}  err={np.linalg.norm(aa_direct - aa_from_mat):.6f}"
)
print(
    f"aa reorder[w,x,y,z]: {aa_reorder} err={np.linalg.norm(aa_reorder - aa_from_mat):.6f}"
)
print(f"aa from mat (truth): {aa_from_mat}")

env.close()

use_delta=False
Controller : OperationalSpaceController
use_delta  : False
input_max  : [1. 1. 1. 1. 1. 1.]
input_min  : [-1. -1. -1. -1. -1. -1.]
output_max : [0.05 0.05 0.05 0.5  0.5  0.5 ]
output_min : [-0.05 -0.05 -0.05 -0.5  -0.5  -0.5 ]

robot0_eef_quat  : [ 9.99596605e-01  2.46212832e-04 -2.84001205e-02 -6.99529583e-06]
aa direct [x,y,z,w]: [ 3.14033934e+00  7.73503869e-04 -8.92220072e-02]  err=5.768062
aa reorder[w,x,y,z]: [ 4.92491889e-04 -5.68078799e-02 -1.39924732e-05] err=3.061861
aa from mat (truth): [-2.19205   -2.1931298  0.0622796]


In [ ]:
from scipy.spatial.transform import Rotation


def relative_rotation(aa1, aa2):
    """Rotation matrix that takes orientation aa1 to aa2."""
    R1 = Rotation.from_rotvec(aa1)
    R2 = Rotation.from_rotvec(aa2)
    return (R2 * R1.inv()).as_matrix()

below code confirmed that goal_ori == action axis angle orientation (well as long as you're within joint limits for that pose) and that this orientation is the ctrl.ee_ori_mat (ctrler frame, instead of robot0_eef_quat bs)

also the orientation between robot0_eef_quat and ctrl.ee_ori_mat is not even fixed, i have no idea whats going on.

In [28]:
env, obs = make_env(delta=False)
home = eef(obs)

target_aa = np.array([-2.0, -2.2, 0.08])
action = np.concatenate([home, target_aa, [1.0]])
# action = obs_to_action(env, obs)  # uses controller's ee_ori_mat directly


R_rel_init = (
    T.quat2mat(obs["robot0_eef_quat"]) @ env.env.robots[0].controller.ee_ori_mat.T
)

frames = [get_frame(obs)]
for i in range(40):
    obs, _, _, _ = env.step(action)
    frames.append(get_frame(obs))

    eff_site_name = "gripper0_grip_site"
    obs["robot0_eef_pos"] = env.sim.data.site_xpos[
        env.sim.model.site_name2id(eff_site_name)
    ]
    obs["robot0_eef_quat"] = T.mat2quat(
        env.sim.data.site_xmat[env.sim.model.site_name2id(eff_site_name)].reshape(3, 3)
    )

    if i % 10 == 0:
        # rel2abs:
        goal_pos = env.env.robots[0].controller.goal_pos
        goal_ori = T.quat2axisangle(T.mat2quat(env.env.robots[0].controller.goal_ori))

        # this is the current orientation. let see if it reaches to goal_ori.
        ctrl_ori_mat = env.env.robots[0].controller.ee_ori_mat
        ctrl_aa = T.quat2axisangle(T.mat2quat(ctrl_ori_mat))

        # robot0_eef_quat bs
        aa_obs = np.asarray(T.quat2axisangle(obs["robot0_eef_quat"])).flatten()

        print(f"{goal_pos=}, {goal_ori=}")
        print(f"{action[3:6]} action orientation")
        print(f"{ctrl_aa=}, {aa_obs=}, diff: {np.max(ctrl_aa - aa_obs)}")


R_rel_final = (
    T.quat2mat(obs["robot0_eef_quat"]) @ env.env.robots[0].controller.ee_ori_mat.T
)
print(np.abs(R_rel_init - R_rel_final).max())

env.close()
show_video(frames, "/tmp/custom.mp4")

use_delta=False
goal_pos=array([-2.08464661e-01, -2.78370650e-14,  1.17327948e+00]), goal_ori=array([-1.9999996 , -2.1999998 ,  0.08000004], dtype=float32)
[-2.   -2.2   0.08] action orientation
ctrl_aa=array([-2.17216   , -2.1939945 ,  0.06257938], dtype=float32), aa_obs=array([-2.17216   , -2.1939945 ,  0.06257938], dtype=float32), diff: 0.0
goal_pos=array([-2.08464661e-01, -2.78370650e-14,  1.17327948e+00]), goal_ori=array([-1.9999996 , -2.1999998 ,  0.08000004], dtype=float32)
[-2.   -2.2   0.08] action orientation
ctrl_aa=array([-1.9991984 , -2.1969135 ,  0.08303914], dtype=float32), aa_obs=array([-1.9991984 , -2.1969135 ,  0.08303914], dtype=float32), diff: 0.0
goal_pos=array([-2.08464661e-01, -2.78370650e-14,  1.17327948e+00]), goal_ori=array([-1.9999996 , -2.1999998 ,  0.08000004], dtype=float32)
[-2.   -2.2   0.08] action orientation
ctrl_aa=array([-1.9971826 , -2.1982656 ,  0.08394065], dtype=float32), aa_obs=array([-1.9971826 , -2.1982656 ,  0.08394065], dtype=float32), diff

In [ ]:
# Test 1 — absolute mode, hold home (err should stay near 0)
env, obs = make_env(delta=False)
home = eef(obs)
action = obs_to_action(env, obs)  # uses controller's ee_ori_mat directly
print(f"home={home}  action={action}")
frames = [get_frame(obs)]
for i in range(40):
    obs, _, _, _ = env.step(action)
    frames.append(get_frame(obs))
    if i % 10 == 0:
        # rel2abs:
        goal_pos = env.env.robots[0].controller.goal_pos
        goal_ori = T.quat2axisangle(T.mat2quat(env.env.robots[0].controller.goal_ori))

        print(f"  t={i:2d}  pos={eef(obs)}  err={np.linalg.norm(eef(obs) - home):.4f}")
        print(f"{goal_pos=}, {goal_ori=}")
        print(f"{action[3:6]} action orientation")
env.close()
show_video(frames, "/tmp/test1_hold_home.mp4")

In [ ]:
# Test 2 — absolute mode, move +5cm in X
env, obs = make_env(delta=False)
home = eef(obs)
target = home.copy()
target[0] += 0.05
action = obs_to_action(env, obs, target_pos=target)
print(f"home={home}  target={target}")
frames = [get_frame(obs)]
for i in range(40):
    obs, _, _, _ = env.step(action)
    frames.append(get_frame(obs))
    if i % 10 == 0:
        print(
            f"  t={i:2d}  pos={eef(obs)}  err={np.linalg.norm(eef(obs) - target):.4f}"
        )
env.close()
show_video(frames, "/tmp/test2_move_x.mp4")

use_delta=False
home=[-2.08464661e-01 -2.78370650e-14  1.17327948e+00]  target=[-1.58464661e-01 -2.78370650e-14  1.17327948e+00]
  t= 0  pos=[-2.03716763e-01  6.86811532e-10  1.17287102e+00]  err=0.0453
  t=10  pos=[-1.55548498e-01  1.84260190e-09  1.17586223e+00]  err=0.0039
  t=20  pos=[-1.57279214e-01  2.41068051e-09  1.17420293e+00]  err=0.0015
  t=30  pos=[-1.59045326e-01  4.21334381e-09  1.17283432e+00]  err=0.0007
